# Phase 6 & 7 — Loading PostgreSQL and SQL reporting

This notebook covers the database half of the project:

* **Phase 6** — load the cleaned tables (`datasets/clean/*.csv`) into the `pokemon`
  PostgreSQL schema and verify integrity.
* **Phase 7** — run the reporting queries from `sql/reports.sql` and review the results.

The schema itself is defined in `sql/schema.sql` (7 tables + 2 views) and the
loader lives in `scripts/load_database.py`.

## 0. Setup

In [1]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from sqlalchemy import create_engine, text

ROOT = Path.cwd()
if (ROOT / "scripts").exists():
    sys.path.insert(0, str(ROOT / "scripts"))

DB_URL = os.environ.get("POKEMON_DB_URL", "postgresql+psycopg2:///pokemon_analysis")
engine = create_engine(DB_URL)

with engine.connect() as c:
    print("server  :", c.execute(text("select version()")).scalar_one())
    print("database:", c.execute(text("select current_database()")).scalar_one())
    print("schema  :", c.execute(text("select current_schema()")).scalar_one())

server  : PostgreSQL 18.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 16.2.1 20260810, 64-bit
database: pokemon_analysis
schema  : pokemon


## 6.1 Load the tables

`scripts/load_database.py` truncates and reloads every table in foreign-key
order, so it is safe to re-run. Here we call it as a subprocess and show its
output.

In [2]:
load = subprocess.run(
    [sys.executable, "scripts/load_database.py"],
    cwd=ROOT, capture_output=True, text=True,
)
print(load.stdout)
if load.returncode != 0:
    print("STDERR:\n", load.stderr)
print("exit code:", load.returncode)

target: postgresql+psycopg2:///pokemon_analysis
truncating tables ...
loading:
  loaded generations             9 rows
  loaded types                  18 rows
  loaded abilities             284 rows
  loaded pokemon              1025 rows
  loaded pokemon_stats        1025 rows
  loaded pokemon_abilities    2411 rows
  loaded type_matchups         324 rows

verification:
  [ok ] generations             9 (expected 9)
  [ok ] types                  18 (expected 18)
  [ok ] abilities             284 (expected 284)
  [ok ] pokemon              1025 (expected 1025)
  [ok ] pokemon_stats        1025 (expected 1025)
  [ok ] pokemon_abilities    2411 (expected 2411)
  [ok ] type_matchups         324 (expected 324)
  [ok ] v_pokemon_profile    1025 (expected 1025)
  [ok ] v_pokemon_type_defense  18450 (expected 18450)
  sample  v_pokemon_profile #9 = ('Blastoise', 530, 'water', 'special', 'medium')
  [ok ] sum(base_total) = 438379 (expected 438379)
  [ok ] dual types = 526 (expected 526)

LOAD

## 6.2 Schema inventory

Tables and views in the `pokemon` schema with their row counts.

In [3]:
tables = pd.read_sql(
    """SELECT table_name, table_type
       FROM information_schema.tables
       WHERE table_schema = 'pokemon'
       ORDER BY table_type, table_name""",
    engine,
)

with engine.connect() as c:
    counts = {
        t: c.execute(text(f"SELECT count(*) FROM pokemon.{t}")).scalar_one()
        for t in tables.loc[tables.table_type == "BASE TABLE", "table_name"]
    }

inventory = tables.assign(rows=tables.table_name.map(counts))
display(inventory)

,table_name,table_type,rows
0,abilities,BASE TABLE,284.0
1,generations,BASE TABLE,9.0
2,pokemon,BASE TABLE,1025.0
3,pokemon_abilities,BASE TABLE,2411.0
4,pokemon_stats,BASE TABLE,1025.0
5,type_matchups,BASE TABLE,324.0
6,types,BASE TABLE,18.0
7,v_pokemon_profile,VIEW,NaN
8,v_pokemon_type_defense,VIEW,NaN


## 6.3 Integrity checks

Each check below must return `0` (or the expected count).

In [4]:
checks = {
    "pokemon rows (expect 1025)": "SELECT count(*) FROM pokemon.pokemon",
    "pokemon_stats rows (expect 1025)": "SELECT count(*) FROM pokemon.pokemon_stats",
    "profile view rows (expect 1025)": "SELECT count(*) FROM pokemon.v_pokemon_profile",
    "defense view rows (expect 18450)": "SELECT count(*) FROM pokemon.v_pokemon_type_defense",
    "type2 == type1 (expect 0)":
        "SELECT count(*) FROM pokemon.pokemon WHERE type2_id = type1_id",
    "base_total mismatch (expect 0)":
        "SELECT count(*) FROM pokemon.pokemon_stats "
        "WHERE base_total <> hp + attack + defense + sp_attack + sp_defense + speed",
    "hidden ability off slot 3 (expect 0)":
        "SELECT count(*) FROM pokemon.pokemon_abilities WHERE is_hidden <> (slot = 3)",
    "pokemon without stats (expect 0)":
        "SELECT count(*) FROM pokemon.pokemon p "
        "LEFT JOIN pokemon.pokemon_stats s USING (pokedex_number) "
        "WHERE s.pokedex_number IS NULL",
}

with engine.connect() as c:
    for label, sql in checks.items():
        print(f"{label:44s} -> {c.execute(text(sql)).scalar_one()}")

pokemon rows (expect 1025)                   -> 1025
pokemon_stats rows (expect 1025)             -> 1025
profile view rows (expect 1025)              -> 1025
defense view rows (expect 18450)             -> 18450
type2 == type1 (expect 0)                    -> 0
base_total mismatch (expect 0)               -> 0
hidden ability off slot 3 (expect 0)         -> 0
pokemon without stats (expect 0)             -> 0


## 6.4 Database vs. source CSV

Confirm the database agrees with `datasets/pokemon_clean.csv`.

In [5]:
clean = pd.read_csv(ROOT / "datasets" / "pokemon_clean.csv")

with engine.connect() as c:
    n_db = c.execute(text("SELECT count(*) FROM pokemon.v_pokemon_profile")).scalar_one()
    tot_db = c.execute(text("SELECT sum(base_total) FROM pokemon.pokemon_stats")).scalar_one()
    dual_db = c.execute(
        text("SELECT count(*) FROM pokemon.v_pokemon_profile WHERE is_dual_type")
    ).scalar_one()

print(f"{'metric':12s} {'CSV':>10s} {'DB':>10s}")
print(f"{'rows':12s} {len(clean):>10d} {n_db:>10d}")
print(f"{'base_total':12s} {int(clean.base_total.sum()):>10d} {tot_db:>10d}")
print(f"{'dual types':12s} {int(clean.is_dual_type.sum()):>10d} {dual_db:>10d}")

metric              CSV         DB
rows               1025       1025
base_total       438379     438379
dual types          526        526


## 7.1 The reporting queries

`sql/reports.sql` holds 12 reports, each introduced by a `-- @report:` marker.
`scripts/run_reports.py` parses those markers, executes the queries and writes
`reports/sql_reports.md`. Here we reuse its helpers to run the same queries
interactively.

In [6]:
from run_reports import split_reports

REPORTS = split_reports((ROOT / "sql" / "reports.sql").read_text())
print(f"{len(REPORTS)} reports found in sql/reports.sql:")
for title, _ in REPORTS:
    print("  -", title)

12 reports found in sql/reports.sql:
  - R1. Pokedex roster and legendary share per generation
  - R2. Top 10 Pokemon by base stat total
  - R3. Average base stat total per generation (power creep)
  - R4. Legendary vs regular stat profile
  - R5. Offensive type power: incoming effectiveness by attacking type
  - R6. Type combinations with the most weaknesses (dual types)
  - R7. Most common abilities
  - R8. Top 10 "glass cannons" (offense minus bulk, speed as tiebreak)
  - R9. Hardest-to-catch strong Pokemon (base_total >= 500)
  - R10. Primary type distribution (share of the dex)
  - R11. Most common type combinations
  - R12. Dual vs single type: average stat profile


## 7.2 Report results

In [7]:
with engine.connect() as conn:
    for title, body in REPORTS:
        display(Markdown(f"### {title}"))
        df = pd.read_sql(text(body), conn)
        display(df)

### R1. Pokedex roster and legendary share per generation

,generation_id,generation,dex_start,dex_end,pokemon,legendary,avg_base_total
0,1,Generation I,1,151,151,5,407.6
1,2,Generation II,152,251,100,6,407.2
2,3,Generation III,252,386,135,10,403.7
3,4,Generation IV,387,493,107,14,445.6
4,5,Generation V,494,649,156,13,425.8
5,6,Generation VI,650,721,72,6,429.3
6,7,Generation VII,722,809,88,23,449.4
7,8,Generation VIII,810,905,96,12,439.2
8,9,Generation IX,906,1025,120,12,457.4


### R2. Top 10 Pokemon by base stat total

,pokedex_number,name,generation_name,type_pair,hp,attack,defense,sp_attack,sp_defense,speed,base_total
0,493,Arceus,Generation IV,normal,120,120,120,120,120,120,720
1,890,Eternatus,Generation VIII,poison / dragon,140,85,95,145,95,130,690
2,150,Mewtwo,Generation I,psychic,106,110,90,154,90,130,680
3,249,Lugia,Generation II,psychic / flying,106,90,130,90,154,110,680
4,484,Palkia,Generation IV,water / dragon,90,120,100,150,120,100,680
5,716,Xerneas,Generation VI,fairy,126,131,95,131,98,99,680
6,717,Yveltal,Generation VI,dark / flying,126,131,95,131,98,99,680
7,792,Lunala,Generation VII,psychic / ghost,137,113,89,137,107,97,680
8,791,Solgaleo,Generation VII,psychic / steel,137,137,107,113,89,97,680
9,384,Rayquaza,Generation III,dragon / flying,105,150,90,150,90,95,680


### R3. Average base stat total per generation (power creep)

...

,primary_type,pokemon,pct_of_dex,legendary,avg_base_total
0,water,134,13.1,7,418.9
1,normal,118,11.5,6,399.8
2,grass,103,10.0,5,413.1
3,bug,83,8.1,3,374.6
4,fire,66,6.4,5,446.2
5,psychic,60,5.9,18,446.7
6,electric,59,5.8,8,436.3
7,rock,58,5.7,4,441.2
8,dark,45,4.4,8,454.7
9,poison,42,4.1,5,426.3


### R11. Most common type combinations

,type_pair,is_dual_type,pokemon,avg_base_total
0,normal,False,75,407.7
1,water,False,74,396.5
2,grass,False,47,395.8
3,psychic,False,39,425.6
4,fire,False,36,403.3
5,electric,False,35,415.8
6,fighting,False,28,412.1
7,normal / flying,True,27,368.1
8,bug,False,23,285.2
9,fairy,False,21,417.8


### R12. Dual vs single type: average stat profile

,typing,pokemon,avg_base_total,avg_offense,avg_bulk,avg_speed
0,dual,526,451.0,156.3,224.3,70.4
1,single,499,403.1,138.4,200.9,63.8


## 7.3 Saved output

The same results are written to `reports/sql_reports.md`.

In [8]:
report_md = ROOT / "reports" / "sql_reports.md"
print(f"{report_md.relative_to(ROOT)}  ({report_md.stat().st_size:,} bytes)")
print(report_md.read_text()[:300] + " ...")

reports/sql_reports.md  (13,791 bytes)
# SQL reports

Generated: 2026-10-16 16:23  
Database: `pokemon_analysis` (schema `pokemon`)  
Reports: 12

Auto-generated by `scripts/run_reports.py` from `sql/reports.sql`.

## R1. Pokedex roster and legendary share per generation

```sql
-- Uses: INNER/LEFT joins, aggregates, FILTER
SELECT g.gene ...


## Key takeaways

* **Legendaries vs regulars** (R4): legendaries average **590.4** base total vs
  **409.9** for regular Pokémon, and own the top of the dex (Arceus 720).
* **Power creep** (R3): average base total climbs from **407.6** (Gen I) to
  **457.4** (Gen IX), a **+49.8** increase, though Gen II–III actually dip below Gen I.
* **Best offensive types** (R5): by average incoming multiplier, **rock (1.231)**
  and **ice (1.204)** hit the most Pokémon super-effectively.
* **Typing matters** (R12): dual-typed Pokémon average **451** base total vs
  **403** for single-typed — but R6 shows some combinations (e.g. `normal / flying`)
  rack up many weaknesses.
* **Abilities** (R7): **Swift Swim** is the most common ability (44 Pokémon, 12 hidden).
* **Dex composition** (R10): **water** is the most common primary type (13.1%),
  while **psychic** contributes the most legendaries (18).